# 🇧🇷 NOTEBOOK 01: EXTRAÇÃO CARTOLA FC (v2)

**Objetivo:** Extrair dados do Cartola FC 2025 (Mercado + Rodadas disponíveis)  
**Estratégia:** Híbrida (Mercado completo + Performance por rodada)  
**Output:** Bronze layer

In [16]:
# Setup
import sys
sys.path.append('../src')

import requests
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import json
from datetime import datetime

from utils.config import CARTOLA_BASE_URL, BRONZE_CARTOLA, CARTOLA_POSITIONS
from utils.logger import setup_logger

logger = setup_logger('cartola', 'logs/cartola.log')

print("✅ Setup completo!")
print(f"🔗 API: {CARTOLA_BASE_URL}")
print(f"📅 Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

✅ Setup completo!
🔗 API: https://api.cartola.globo.com
📅 Data: 04/02/2026 14:20


---
## 🧪 1. TESTE RÁPIDO DA API

In [17]:
print("🧪 Testando conexão com API...\n")

try:
    resp = requests.get(f"{CARTOLA_BASE_URL}/atletas/mercado", timeout=10)
    
    if resp.status_code == 200:
        data = resp.json()
        num_atletas = len(data.get('atletas', {}))
        
        print("✅ API FUNCIONANDO!")
        print(f"   Atletas no mercado: {num_atletas}")
        print(f"   Clubes: {len(data.get('clubes', {}))}")
        print(f"   Posições: {len(data.get('posicoes', {}))}\n")
        
        logger.info(f"API OK - {num_atletas} atletas disponíveis")
    else:
        print(f"❌ Erro: Status {resp.status_code}")

except Exception as e:
    print(f"❌ Erro na conexão: {e}")
    logger.error(f"Falha no teste: {e}")

🧪 Testando conexão com API...

✅ API FUNCIONANDO!
   Atletas no mercado: 692
   Clubes: 20
   Posições: 6

2026-02-04 14:20:25 | cartola | INFO | API OK - 692 atletas disponíveis


---
## 📥 2. EXTRAÇÃO 1: MERCADO COMPLETO

In [18]:
print("="*70)
print(" "*20 + "📊 EXTRAÇÃO: MERCADO")
print("="*70 + "\n")

print("🔍 Buscando mercado completo...\n")

try:
    resp = requests.get(f"{CARTOLA_BASE_URL}/atletas/mercado", timeout=10)
    
    if resp.status_code == 200:
        data_mercado = resp.json()
        
        print(f"🔍 Tipo de resposta: {type(data_mercado)}")
        
        if isinstance(data_mercado, dict) and 'atletas' in data_mercado:
            
            atletas_raw = data_mercado['atletas']
            print(f"🔍 Tipo de 'atletas': {type(atletas_raw)}")
            
            # CASO A: atletas é um DICIONÁRIO {id: {...}, id: {...}}
            if isinstance(atletas_raw, dict):
                print("✅ Estrutura: Dicionário de atletas\n")
                
                atletas_mercado = []
                
                for atleta_id, atleta in atletas_raw.items():
                    atleta['atleta_id'] = int(atleta_id)
                    atleta['fonte'] = 'mercado'
                    atletas_mercado.append(atleta)
                
                df_mercado = pd.DataFrame(atletas_mercado)
            
            # CASO B: atletas é uma LISTA [{...}, {...}]
            elif isinstance(atletas_raw, list):
                print("✅ Estrutura: Lista de atletas\n")
                
                df_mercado = pd.DataFrame(atletas_raw)
                
                # Garantir atleta_id
                if 'atleta_id' not in df_mercado.columns:
                    if 'id' in df_mercado.columns:
                        df_mercado['atleta_id'] = df_mercado['id']
                    else:
                        df_mercado['atleta_id'] = range(len(df_mercado))
                
                df_mercado['fonte'] = 'mercado'
            
            else:
                print(f"❌ Tipo inesperado de 'atletas': {type(atletas_raw)}")
                df_mercado = pd.DataFrame()
        
        else:
            print(f"❌ Resposta sem chave 'atletas'")
            df_mercado = pd.DataFrame()
        
        # Processar se extraiu com sucesso
        if not df_mercado.empty:
            print(f"✅ Mercado extraído: {len(df_mercado):,} atletas")
            print(f"📋 Colunas: {len(df_mercado.columns)}")
            print(f"📋 Primeiras colunas: {df_mercado.columns.tolist()[:10]}\n")
            
            # Adicionar nome da posição
            if 'posicao_id' in df_mercado.columns:
                df_mercado['posicao_nome'] = df_mercado['posicao_id'].map(CARTOLA_POSITIONS)
                print(f"📊 Distribuição por posição:")
                print(df_mercado['posicao_nome'].value_counts())
            
            # Estatísticas
            numeric_cols = df_mercado.select_dtypes(include=[np.number]).columns
            
            if 'pontos_num' in numeric_cols:
                print(f"\n📊 Estatísticas de pontos:")
                print(f"   Média: {df_mercado['pontos_num'].mean():.2f}")
                print(f"   Máximo: {df_mercado['pontos_num'].max():.2f}")
            elif 'media_num' in numeric_cols:
                print(f"\n📊 Estatísticas de média:")
                print(f"   Média geral: {df_mercado['media_num'].mean():.2f}")
            
            if 'preco_num' in numeric_cols:
                print(f"\n📊 Estatísticas de preço:")
                print(f"   Média: C$ {df_mercado['preco_num'].mean():.2f}")
                print(f"   Máximo: C$ {df_mercado['preco_num'].max():.2f}")
            
            logger.info(f"Mercado extraído: {len(df_mercado)} atletas")
        else:
            print("❌ Não foi possível extrair atletas")
    
    else:
        print(f"❌ Status HTTP: {resp.status_code}")
        df_mercado = pd.DataFrame()

except Exception as e:
    print(f"❌ Erro: {e}")
    import traceback
    print(f"\n🔍 Traceback completo:")
    print(traceback.format_exc())
    df_mercado = pd.DataFrame()
    logger.error(f"Erro ao extrair mercado: {e}")

print("\n" + "="*70)

                    📊 EXTRAÇÃO: MERCADO

🔍 Buscando mercado completo...

🔍 Tipo de resposta: <class 'dict'>
🔍 Tipo de 'atletas': <class 'list'>
✅ Estrutura: Lista de atletas

✅ Mercado extraído: 692 atletas
📋 Colunas: 18
📋 Primeiras colunas: ['scout', 'jogos_num', 'atleta_id', 'rodada_id', 'clube_id', 'posicao_id', 'status_id', 'pontos_num', 'media_num', 'variacao_num']

📊 Distribuição por posição:
posicao_nome
MEI    217
ATA    171
ZAG    113
LAT    105
GOL     66
TEC     20
Name: count, dtype: int64

📊 Estatísticas de pontos:
   Média: 1.75
   Máximo: 17.20

📊 Estatísticas de preço:
   Média: C$ 5.52
   Máximo: C$ 20.09
2026-02-04 14:20:25 | cartola | INFO | Mercado extraído: 692 atletas



---
## ⚽ 3. EXTRAÇÃO 2: RODADAS DISPONÍVEIS

In [19]:
print("="*70)
print(" "*15 + "📊 EXTRAÇÃO: RODADAS (DETECÇÃO AUTOMÁTICA)")
print("="*70 + "\n")

print("🔍 Detectando rodadas disponíveis...\n")

atletas_rodadas = []
rodadas_sucesso = []
max_404_consecutivos = 3
consecutivos_404 = 0

for rodada in range(1, 39):
    
    url = f"{CARTOLA_BASE_URL}/atletas/pontuados/{rodada}"
    
    try:
        resp = requests.get(url, timeout=10)
        
        if resp.status_code == 200:
            data = resp.json()
            
            if 'atletas' in data and len(data['atletas']) > 0:
                
                # Extrair atletas da rodada
                for atleta_id, atleta in data['atletas'].items():
                    atleta['atleta_id'] = int(atleta_id)
                    atleta['rodada'] = rodada
                    atleta['fonte'] = 'rodada'
                    atletas_rodadas.append(atleta)
                
                rodadas_sucesso.append(rodada)
                consecutivos_404 = 0
                
                print(f"✅ Rodada {rodada}: {len(data['atletas'])} atletas")
                logger.info(f"Rodada {rodada}: {len(data['atletas'])} atletas")
            
            else:
                print(f"⚠️  Rodada {rodada}: sem dados")
                consecutivos_404 += 1
        
        elif resp.status_code == 400:
            # Bad Request = rodada ainda não processada
            consecutivos_404 += 1
        
        elif resp.status_code == 404:
            consecutivos_404 += 1
        
        # Parar após 3 rodadas consecutivas sem dados
        if consecutivos_404 >= max_404_consecutivos:
            print(f"\n⚠️  Parando: {consecutivos_404} rodadas consecutivas não encontradas")
            break
        
        time.sleep(1)  # Rate limit
    
    except Exception as e:
        print(f"❌ Rodada {rodada}: erro - {e}")
        consecutivos_404 += 1
        
        if consecutivos_404 >= max_404_consecutivos:
            break

# Consolidar
if atletas_rodadas:
    df_rodadas = pd.DataFrame(atletas_rodadas)
    
    print(f"\n✅ Total extraído: {len(df_rodadas):,} registros")
    print(f"✅ Rodadas disponíveis: {rodadas_sucesso}")
    print(f"✅ Atletas únicos: {df_rodadas['atleta_id'].nunique():,}")
    
    # Adicionar nome da posição
    if 'posicao_id' in df_rodadas.columns:
        df_rodadas['posicao_nome'] = df_rodadas['posicao_id'].map(CARTOLA_POSITIONS)
    
    logger.info(f"Rodadas extraídas: {len(df_rodadas)} registros, {len(rodadas_sucesso)} rodadas")

else:
    print("\n⚠️  Nenhuma rodada disponível")
    df_rodadas = pd.DataFrame()

print("\n" + "="*70)

               📊 EXTRAÇÃO: RODADAS (DETECÇÃO AUTOMÁTICA)

🔍 Detectando rodadas disponíveis...

✅ Rodada 1: 336 atletas
2026-02-04 14:20:26 | cartola | INFO | Rodada 1: 336 atletas

⚠️  Parando: 3 rodadas consecutivas não encontradas

✅ Total extraído: 336 registros
✅ Rodadas disponíveis: [1]
✅ Atletas únicos: 336
2026-02-04 14:20:29 | cartola | INFO | Rodadas extraídas: 336 registros, 1 rodadas



---
## 🔗 4. COMBINAR DADOS (MERCADO + RODADAS)

In [20]:
print("="*70)
print(" "*20 + "🔗 COMBINANDO DADOS")
print("="*70 + "\n")

if not df_mercado.empty or not df_rodadas.empty:
    
    # Estratégia: usar mercado como base, enriquecer com rodadas
    
    if not df_mercado.empty and not df_rodadas.empty:
        print("✅ Combinando mercado + rodadas...\n")
        
        # Manter mercado como base (dados agregados da temporada)
        df_final = df_mercado.copy()
        
        # Adicionar estatísticas das rodadas (se houver múltiplas)
        if len(rodadas_sucesso) > 0:
            
            # Calcular média por atleta nas rodadas
            if 'pontuacao' in df_rodadas.columns:
                rodadas_stats = df_rodadas.groupby('atleta_id')['pontuacao'].agg([
                    ('pontos_rodadas_mean', 'mean'),
                    ('pontos_rodadas_std', 'std'),
                    ('num_rodadas_jogou', 'count')
                ]).reset_index()
                
                # Merge com mercado
                df_final = df_final.merge(rodadas_stats, on='atleta_id', how='left')
                
                print(f"✅ Estatísticas de rodadas adicionadas ao mercado")
        
        print(f"\n📊 Dataset final (MERCADO enriquecido):")
        print(f"   Registros: {len(df_final):,}")
        print(f"   Colunas: {len(df_final.columns)}")
    
    elif not df_mercado.empty:
        print("✅ Usando apenas MERCADO\n")
        df_final = df_mercado.copy()
        
        print(f"📊 Dataset final:")
        print(f"   Registros: {len(df_final):,}")
    
    elif not df_rodadas.empty:
        print("✅ Usando apenas RODADAS\n")
        df_final = df_rodadas.copy()
        
        print(f"📊 Dataset final:")
        print(f"   Registros: {len(df_final):,}")
    
    # Renomear coluna de pontos se necessário
    if 'pontuacao' in df_final.columns and 'pontos_num' not in df_final.columns:
        df_final = df_final.rename(columns={'pontuacao': 'pontos_num'})
        print(f"   ✅ Coluna 'pontuacao' → 'pontos_num'")
    
    logger.info(f"Dataset final: {len(df_final)} registros")

else:
    print("❌ Nenhum dado foi extraído!")
    df_final = pd.DataFrame()

print("\n" + "="*70)

                    🔗 COMBINANDO DADOS

✅ Combinando mercado + rodadas...

✅ Estatísticas de rodadas adicionadas ao mercado

📊 Dataset final (MERCADO enriquecido):
   Registros: 692
   Colunas: 22
2026-02-04 14:20:29 | cartola | INFO | Dataset final: 692 registros



---
## 📊 5. ANÁLISE RÁPIDA

In [21]:
if not df_final.empty:
    
    print("="*70)
    print(" "*20 + "📊 ANÁLISE DOS DADOS")
    print("="*70 + "\n")
    
    print(f"📋 INFORMAÇÕES GERAIS:")
    print(f"   Total de atletas: {len(df_final):,}")
    print(f"   Colunas: {len(df_final.columns)}")
    
    if 'posicao_nome' in df_final.columns:
        print(f"\n📊 ATLETAS POR POSIÇÃO:")
        print(df_final['posicao_nome'].value_counts())
    
    if 'pontos_num' in df_final.columns:
        print(f"\n📊 ESTATÍSTICAS DE PONTOS:")
        print(df_final['pontos_num'].describe().round(2))
        
        # Top 10
        if 'apelido' in df_final.columns:
            print(f"\n🏆 TOP 10 PONTUADORES:")
            top10 = df_final.nlargest(10, 'pontos_num')[['apelido', 'posicao_nome', 'pontos_num']]
            print(top10.to_string(index=False))
    
    if 'preco_num' in df_final.columns:
        print(f"\n💰 ESTATÍSTICAS DE PREÇO:")
        print(f"   Média: C$ {df_final['preco_num'].mean():.2f}")
        print(f"   Mediana: C$ {df_final['preco_num'].median():.2f}")
        print(f"   Máximo: C$ {df_final['preco_num'].max():.2f}")
    
    print("\n" + "="*70)

else:
    print("❌ Sem dados para analisar!")

                    📊 ANÁLISE DOS DADOS

📋 INFORMAÇÕES GERAIS:
   Total de atletas: 692
   Colunas: 22

📊 ATLETAS POR POSIÇÃO:
posicao_nome
MEI    217
ATA    171
ZAG    113
LAT    105
GOL     66
TEC     20
Name: count, dtype: int64

📊 ESTATÍSTICAS DE PONTOS:
count    692.00
mean       1.75
std        3.24
min       -6.00
25%        0.00
50%        0.00
75%        2.27
max       17.20
Name: pontos_num, dtype: float64

🏆 TOP 10 PONTUADORES:
         apelido posicao_nome  pontos_num
          Danilo          MEI        17.2
  Gabriel Menino          MEI        16.7
Juninho Capixaba          LAT        16.2
     Flaco López          ATA        16.1
         Luciano          ATA        15.3
     Breno Bidon          MEI        14.7
    Lucho Acosta          MEI        13.8
   Higor Meritão          MEI        13.4
     Jean Carlos          MEI        13.0
         Barreal          MEI        12.6

💰 ESTATÍSTICAS DE PREÇO:
   Média: C$ 5.52
   Mediana: C$ 5.00
   Máximo: C$ 20.09



---
## 💾 6. SALVAR (BRONZE LAYER)

In [22]:
if not df_final.empty:
    
    print("="*70)
    print(" "*20 + "💾 SALVANDO - BRONZE LAYER")
    print("="*70 + "\n")
    
    # Criar diretório
    BRONZE_CARTOLA.mkdir(parents=True, exist_ok=True)
    
    # Salvar Parquet
    output_parquet = BRONZE_CARTOLA / 'atletas_all.parquet'
    df_final.to_parquet(output_parquet, compression='snappy', index=False)
    
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    
    print(f"✅ DADOS SALVOS:")
    print(f"   📁 {output_parquet}")
    print(f"   📊 {len(df_final):,} registros")
    print(f"   📋 {len(df_final.columns)} colunas")
    print(f"   📦 {size_mb:.2f} MB")
    
    # Salvar CSV (amostra)
    output_csv = BRONZE_CARTOLA / 'atletas_sample.csv'
    df_final.head(500).to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"\n✅ Amostra CSV: atletas_sample.csv (500 primeiras linhas)")
    
    logger.info(f"Dados salvos: {len(df_final)} registros, {size_mb:.2f} MB")
    
    print("\n" + "="*70)

else:
    print("❌ Sem dados para salvar!")

                    💾 SALVANDO - BRONZE LAYER

✅ DADOS SALVOS:
   📁 D:\football analytics project\data\bronze\cartola\atletas_all.parquet
   📊 692 registros
   📋 22 colunas
   📦 0.06 MB

✅ Amostra CSV: atletas_sample.csv (500 primeiras linhas)
2026-02-04 14:20:29 | cartola | INFO | Dados salvos: 692 registros, 0.06 MB



---
## 📋 7. RESUMO FINAL

In [23]:
print("\n" + "="*70)
print(" "*15 + "🎉 NOTEBOOK 01 - EXTRAÇÃO CONCLUÍDA!")
print("="*70 + "\n")

if not df_final.empty:
    print("📊 RESUMO:\n")
    
    print(f"✅ Fonte MERCADO: {len(df_mercado) if not df_mercado.empty else 0:,} atletas")
    print(f"✅ Fonte RODADAS: {len(df_rodadas) if not df_rodadas.empty else 0:,} registros")
    print(f"✅ Dataset final: {len(df_final):,} atletas")
    
    if rodadas_sucesso:
        print(f"\n✅ Rodadas extraídas: {rodadas_sucesso}")
    
    print(f"\n📁 Arquivo salvo:")
    print(f"   📂 {BRONZE_CARTOLA}/atletas_all.parquet")
    
    print(f"\n🎯 PRÓXIMO PASSO:")
    print(f"   Notebook 03: Data Quality & Cleaning")

else:
    print("❌ Nenhum dado foi extraído!")
    print("\n📋 Possíveis causas:")
    print("   1. API do Cartola fora do ar")
    print("   2. Problemas de conexão")
    print("   3. Temporada ainda não iniciou")

print("\n" + "="*70)

logger.info("Notebook 01 concluído")


               🎉 NOTEBOOK 01 - EXTRAÇÃO CONCLUÍDA!

📊 RESUMO:

✅ Fonte MERCADO: 692 atletas
✅ Fonte RODADAS: 336 registros
✅ Dataset final: 692 atletas

✅ Rodadas extraídas: [1]

📁 Arquivo salvo:
   📂 D:\football analytics project\data\bronze\cartola/atletas_all.parquet

🎯 PRÓXIMO PASSO:
   Notebook 03: Data Quality & Cleaning

2026-02-04 14:20:29 | cartola | INFO | Notebook 01 concluído
